In [ ]:
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

import pandas as pd
import requests
import time
import re

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "01_craw_data").is_dir():
    for parent in PROJECT_DIR.parents:
        if (parent / "01_craw_data").is_dir():
            PROJECT_DIR = parent
            break

CATEGORIES_DIR = PROJECT_DIR / "01_craw_data" / "category"
CATEGORIES_DIR.mkdir(parents=True, exist_ok=True)

CATEGORIES_FILE = CATEGORIES_DIR / "category.csv"
SUB_CATEGORY_FILE = CATEGORIES_DIR / "sub_category.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://tiki.vn/"
}

# 1. Categories

In [ ]:
url = "https://api.tiki.vn/raiden/v2/menu-config"
params = {"platform": "desktop"}

with requests.Session() as session:
    session.headers.update(HEADERS)
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    items = response.json().get("menu_block", {}).get("items", [])

categories = []
for item in items:
    parts = urlparse(item.get("link") or "").path.strip("/").split("/")

    if len(parts) >= 2 and re.fullmatch(r"c\d+", parts[-1]):
        cid = parts[-1][1:]
        categories.append({
            "category_id": int(cid),
            "name": item.get("text"),
            "urlKey": parts[-2],
            "src": parts[-1],
        })

if not categories:
    raise ValueError("Không lấy được danh mục nào, kiểm tra lại API menu-config.")

df_categories = pd.DataFrame(categories).drop_duplicates("category_id")
df_categories.to_csv(CATEGORIES_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu {len(df_categories)} danh mục vào {CATEGORIES_FILE}")

# 2. Sub_Category

In [ ]:
# Cấu hình
SAVE_EVERY = 100
REQUEST_DELAY = 0.5

if not CATEGORIES_FILE.exists():
    raise FileNotFoundError(f"Không tìm thấy {CATEGORIES_FILE}. Hãy chạy phần Crawl Categories trước.")

df_roots = pd.read_csv(CATEGORIES_FILE, low_memory=False)
id_col = "category_id" if "category_id" in df_roots.columns else "id"
df_roots = df_roots.drop_duplicates(id_col)

def save_data(rows):
    if not rows:
        return

    new_data = pd.DataFrame(rows)

    if SUB_CATEGORY_FILE.exists():
        old_data = pd.read_csv(SUB_CATEGORY_FILE, low_memory=False)
        data = pd.concat([old_data, new_data], ignore_index=True, sort=False)
    else:
        data = new_data

    data = data.drop_duplicates(keep="last")
    data.to_csv(SUB_CATEGORY_FILE, index=False, encoding="utf-8-sig")
    rows.clear()

def get_children(session, parent_id):
    try:
        response = session.get(
            "https://tiki.vn/api/v2/categories",
            params={"include": "children", "parent_id": parent_id},
            timeout=20
        )
        response.raise_for_status()
        data = response.json()
        return data.get("data", data.get("children", []))
    except Exception as e:
        print(f"Lỗi category_id={parent_id}: {e}")
        return None

def dfs(session, root_id, root_name, parent_id, path, rows):
    global total_count
    children = get_children(session, parent_id)

    if children is None:
        return False

    if not children:
        if path:
            row = {"category_id": root_id, "category_name": root_name}
            for level, item in enumerate(path, 1):
                row[f"sub_category_{level}_id"] = item["id"]
                row[f"sub_category_{level}"] = item["name"]
            rows.append(row)
            total_count += 1

            if len(rows) >= SAVE_EVERY:
                save_data(rows)
        return True

    success = True
    for child in children:
        if child.get("id") is None:
            continue

        ok = dfs(session, root_id, root_name, child["id"],
            path + [{"id": child["id"], "name": child.get("name")}], rows
        )
        success = success and ok
        time.sleep(REQUEST_DELAY)
    return success

total_count = 0
failed_roots = []
rows = []

with requests.Session() as session:
    session.headers.update(HEADERS)
    for _, root in df_roots.iterrows():
        start_count = total_count
        root_id = int(root[id_col])
        root_name = root["name"]
        success = dfs(session, root_id, root_name, root_id, [], rows)

        if not success:
            failed_roots.append(root_name)
            print(f"Crawl chưa hoàn tất: {root_name} ({total_count - start_count} nhánh lá đã lấy)")
        elif total_count > start_count:
            print(f"Crawl: {root_name} - {total_count - start_count} sub_categories")
        else:
            print(f"Không tìm thấy danh mục con: {root_name}")

save_data(rows)

print(f"Hoàn tất crawl sub-categories. Tổng nhánh lá: {total_count}")

if failed_roots:
    print("Các danh mục cần kiểm tra/crawl lại:", failed_roots)